In [1]:
def main(datasources, start_date, end_date):
    """
    因子构建主函数（XGBoost v9: close_lag + 量价交互）

    ⚠️ 赛制约定（重要）:
        评测时平台只会替换 datasources / start_date / end_date 三个入参,且 start_date~end_date
        指向的是【测试集区间】。因此训练区间必须在代码里写死（见下方 TRAIN_START/TRAIN_END）,
        本函数只用写死的训练区间拟合模型,再用平台传入的测试区间做【样本外预测】。
        切勿用传入的 start_date/end_date 训练模型——那等于在测试集上训练,既是数据泄漏, 也无法体现因子真实的样本外能力。
        后期会审查这类问题

    参数:
        datasources (dict): 数据源表名映射 {逻辑名: 物理表名}，平台会在公榜/私榜自动切换。
                            可用逻辑名: "bar1m" -> 分钟 K 线表, "financial" -> 财务数据表
        start_date (str): 测试集开始时间（平台注入）
        end_date (str):   测试集结束时间（平台注入）

    返回:
        pd.DataFrame: 因子数据，须包含三列 ['date', 'instrument', 'factor']，且不含 inf
    """
    import time
    import json
    import numpy as np
    import pandas as pd
    import dai
    import xgboost as xgb
    import structlog

    logger = structlog.get_logger()

    # ============================================================
    # 训练区间: 写死, 不随平台入参变化
    # ============================================================
    TRAIN_START = '2022-01-01 00:00:00'
    TRAIN_END   = '2023-12-31 23:59:59'

    # v8 原版 13 close_lag
    CLOSE_LAGS = [1, 2, 3, 5, 10, 20, 27, 31, 42, 54, 55, 57, 59]

    # v9 新增 5 个量价交互特征
    VOL_FEATURES = [
        'vp_signed_1',    # sign(close_lag_1-1)  * (vol/vol.shift(1))    T-1 量价协同
        'vp_signed_5',    # sign(close_lag_5-1)  * (vol/vol.shift(5))    5 日量价协同
        'vp_signed_10',   # sign(close_lag_10-1) * (vol/vol.shift(10))   10 日量价协同
        'amt_ratio_5',    # amt.shift(5)/amt     成交额滞后比 (amt=vol*price, 自带量价信息)
        'vol_rel_5_20',   # vol.rolling(5).mean()/vol.rolling(20).mean()  短中期相对均量
    ]

    feature_cols = [f'close_lag_{lag}' for lag in CLOSE_LAGS] + VOL_FEATURES

    def _py(v):
        if hasattr(v, 'item'):
            return v.item()
        return float(v) if isinstance(v, (np.floating, np.integer)) else v

    # ============================================================
    # 特征工程: close 滞后比 + 量价交互 + 标签
    # ============================================================
    def build_features(bar1m_table, sd, ed):
        t0 = time.time()
        logger.info("build_features 开始", start=str(sd), end=str(ed))

        # ----- 量价数据: close 用 ARG_MAX, volume/amount 用 SUM -----
        # v8 只取 close 用 ARG_MAX(close, date) 取日终收盘;
        # vol/amt 必须用 SUM 聚合全天, 若用 ARG_MAX 会只拿日终 1m bar 那一根 (bug)
        t1 = time.time()
        price_start = pd.to_datetime(sd) - pd.Timedelta(days=90)
        price_sql = f"""
            SELECT
                date_trunc('day', date)::DATE AS trading_day,
                instrument,
                ARG_MAX(close, date) AS close,
                SUM(volume) AS volume,
                SUM(amount) AS amount
            FROM {bar1m_table}
            GROUP BY trading_day, instrument
            ORDER BY trading_day, instrument
        """
        price = dai.query(price_sql, filters={'date': [price_start, ed]}).df()
        price = price.rename(columns={'trading_day': 'date'})
        price['date'] = pd.to_datetime(price['date'])
        price = price.sort_values(['instrument', 'date']).reset_index(drop=True)
        logger.info("量价数据查询完成", rows=len(price),
                    n_instruments=price['instrument'].nunique(),
                    elapsed=round(time.time() - t1, 2))

        # 数值清洗
        for col in ['close', 'volume', 'amount']:
            price[col] = pd.to_numeric(price[col], errors='coerce')
            price[col] = price[col].replace([np.inf, -np.inf], np.nan)

        # 停牌/无成交 -> nan (避免 0 除法污染)
        price['volume'] = price['volume'].replace(0, np.nan)
        price['amount'] = price['amount'].replace(0, np.nan)

        # ----- 标签: 未来 1 日收益 (与 v8 一致) -----
        g = price.groupby('instrument', group_keys=False)['close']
        price['label'] = g.shift(-1) / price['close'] - 1
        price['label'] = price['label'].replace([np.inf, -np.inf], np.nan)

        # ----- Alpha360 close 滞后特征 (与 v8 一致) -----
        grp = price.groupby('instrument')
        for lag in CLOSE_LAGS:
            c = f'close_lag_{lag}'
            price[c] = grp['close'].shift(lag) / price['close']
            price[c] = price[c].replace([np.inf, -np.inf], np.nan)

        # ----- v9 新增: 量价交互特征 -----
        # 滞后量
        price['vol_shift_1']  = grp['volume'].shift(1)
        price['vol_shift_5']  = grp['volume'].shift(5)
        price['vol_shift_10'] = grp['volume'].shift(10)
        price['amt_shift_5']  = grp['amount'].shift(5)
        price['vol_roll_5']   = grp['volume'].transform(lambda s: s.rolling(5).mean())
        price['vol_roll_20']  = grp['volume'].transform(lambda s: s.rolling(20).mean())

        # 量价协同: 涨跌方向 * 放量倍数
        # 把 "放量上涨" 和 "放量下跌" 区分开, 解决线性相关被正负抵消的问题
        price['vp_signed_1']  = np.sign(price['close_lag_1']  - 1) * (price['volume'] / price['vol_shift_1'])
        price['vp_signed_5']  = np.sign(price['close_lag_5']  - 1) * (price['volume'] / price['vol_shift_5'])
        price['vp_signed_10'] = np.sign(price['close_lag_10'] - 1) * (price['volume'] / price['vol_shift_10'])

        # 成交额滞后比 (与 close_lag 同构, amount 已隐含价格信息)
        price['amt_ratio_5'] = price['amt_shift_5'] / price['amount']

        # 短中期相对均量 (近期是否异常放量)
        price['vol_rel_5_20'] = price['vol_roll_5'] / price['vol_roll_20']

        # 清理中间列
        price = price.drop(columns=['vol_shift_1', 'vol_shift_5', 'vol_shift_10',
                                    'amt_shift_5', 'vol_roll_5', 'vol_roll_20'])

        for c in VOL_FEATURES:
            price[c] = price[c].replace([np.inf, -np.inf], np.nan)

        # 截断到目标区间
        price = price[(price['date'] >= pd.to_datetime(sd)) & (price['date'] <= pd.to_datetime(ed))]
        logger.info("build_features 结束", rows=len(price),
                    total_elapsed=round(time.time() - t0, 2))
        return price.reset_index(drop=True)

    # ============================================================
    # 第 1 步: 用写死的训练区间拟合模型
    # ============================================================
    logger.info("开始构建训练集", train_start=TRAIN_START, train_end=TRAIN_END)
    train_df = build_features('bigalpha_2026_stock_bar1m', TRAIN_START, TRAIN_END)
    train_df['label'] = train_df['label'].replace([np.inf, -np.inf], np.nan)
    train_df = train_df.dropna(subset=['label'])
    logger.info("训练集构建完成", samples=len(train_df))

    # 验证集: 写死截止日 (2023-10-01, 最后 3 个月), 不用动态比例——确保确定性
    val_cutoff = pd.to_datetime('2023-10-01')
    val_mask = train_df['date'] >= val_cutoff
    train_mask = ~val_mask
    logger.info("验证集切分", train_s=int(train_mask.sum()), val_s=int(val_mask.sum()),
                cutoff_date='2023-10-01')

    # 模型训练 (参数与 v8 完全一致)
    t_fit = time.time()
    model = xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.0421,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=50,
        reg_alpha=5.0, reg_lambda=10.0,
        tree_method='hist', nthread=1, random_state=42,
        early_stopping_rounds=30,
    )
    model.fit(
        train_df.loc[train_mask, feature_cols], train_df.loc[train_mask, 'label'],
        eval_set=[(train_df.loc[val_mask, feature_cols], train_df.loc[val_mask, 'label'])],
        verbose=False,
    )
    logger.info("模型训练完成", elapsed=round(time.time() - t_fit, 2),
                best_iter=model.best_iteration)

    # ============================================================
    # 探针: 保存诊断数据到 JSON
    # ============================================================
    train_pred = model.predict(train_df[feature_cols])
    train_ic = float(np.corrcoef(train_pred, train_df['label'])[0, 1])
    logger.info("训练集 IC", train_ic=round(train_ic, 5),
                pred_std=round(float(np.std(train_pred)), 6),
                label_std=round(float(train_df['label'].std()), 6))

    # 特征重要性
    imp = sorted(zip(feature_cols, model.feature_importances_), key=lambda x: -x[1])
    imp_str = ", ".join([f"{k}:{v:.4f}" for k, v in imp])
    logger.info("特征重要性", top_features=imp_str)

    # 特征-标签相关 (训练集): Pearson + Spearman
    feat_label_corr = {}
    feat_label_spearman = {}
    for c in feature_cols:
        valid = train_df[[c, 'label']].dropna()
        if len(valid) >= 10:
            feat_label_corr[c] = round(_py(valid[c].corr(valid['label'])), 5)
            feat_label_spearman[c] = round(_py(valid[c].rank().corr(valid['label'].rank())), 5)

    # 保存诊断数据
    probe = {
        "version": "v9",
        "train_samples": int(len(train_df)),
        "train_ic": round(train_ic, 5),
        "pred_train_std": round(float(np.std(train_pred)), 6),
        "label_std": round(float(train_df['label'].std()), 6),
        "feature_importance": {k: round(_py(v), 6) for k, v in imp},
        "feature_label_corr": feat_label_corr,
        "feature_label_spearman": feat_label_spearman,
    }
    with open("xgb_v9_probe_data.json", "w") as f:
        json.dump(probe, f, indent=2, ensure_ascii=False)
    logger.info("探针数据已保存到 xgb_v9_probe_data.json")

    # ============================================================
    # 第 2 步: 用平台传入的测试区间做样本外预测
    # ============================================================
    logger.info("开始构建测试集并预测")
    bar1m_table = datasources['bar1m']
    test_df = build_features(bar1m_table, start_date, end_date)
    test_df['factor'] = model.predict(test_df[feature_cols])
    logger.info("测试集预测完成", pred_std=round(float(test_df['factor'].std()), 6))

    # ============================================================
    # 第 3 步: 对齐中证 1000 成分股并输出
    # ============================================================
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)

    result = pd.merge(test_df[['date', 'instrument', 'factor']], stk_pool,
                      how='inner', on=['date', 'instrument'])
    result['factor'] = result['factor'].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=['factor']).reset_index(drop=True)[['date', 'instrument', 'factor']]
    logger.info("因子构建完成", rows=len(result))
    return result


if __name__ == '__main__':
    from bigmodule import M
    import structlog
    import dai
    logger = structlog.get_logger()

    # 本地自测时自行构造数据源映射 (评测时由平台注入, 逻辑名固定为 "bar1m"/"financial")
    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m',
        'financial': 'bigalpha_2026_financial'
    }
    # 本地用这段区间模拟「平台注入的测试集区间」(训练区间已在 main 内写死)
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    # 计算因子
    logger.info(f"计算因子, 测试区间: {start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    # 读取平台因子库用于回归评估, 您可以换成自己的因子库
    logger.info(f"读取因子库, 区间: {start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    # 评估系统
    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )


[2026-07-12 21:40:19] [info     ] 计算因子, 测试区间: 2024-01-01 00:00:00 ~ 2024-12-31 23:59:59
[2026-07-12 21:40:20] [info     ] 开始构建训练集                        train_end='2023-12-31 23:59:59' train_start='2022-01-01 00:00:00'
[2026-07-12 21:40:20] [info     ] build_features 开始              end='2023-12-31 23:59:59' start='2022-01-01 00:00:00'
[2026-07-12 21:40:45] [info     ] 量价数据查询完成                       elapsed=24.35 n_instruments=2214 rows=1151470
[2026-07-12 21:40:48] [info     ] build_features 结束              rows=1027068 total_elapsed=27.84
[2026-07-12 21:40:48] [info     ] 训练集构建完成                        samples=1021345
[2026-07-12 21:40:48] [info     ] 验证集切分                          cutoff_date=2023-10-01 train_s=892998 val_s=128347
[2026-07-12 21:41:47] [info     ] 模型训练完成                         best_iter=295 elapsed=58.91
